# Building a Medical Expert System Using Rules
## Forward Chaining, Backward Chaining and Explanation

### Practical Implementation in Python

**Objective:** Build a simple rule-based expert system that suggests a diagnosis from a patient's symptoms. Implement **Forward Chaining** and **Backward Chaining**, then compare both on:

1. Which conclusions they reach
2. Number of rules examined
3. Execution time

> This notebook is designed to be easy for students to understand. Run the cells from top to bottom.

# 1. Learning Objectives

After completing this practical, you should be able to:

- Explain what an **expert system** is and name its three main parts.
- Represent knowledge as **facts** and **IF-THEN rules**.
- Explain the difference between **data-driven** and **goal-driven** reasoning.
- Implement Forward Chaining from scratch.
- Implement Backward Chaining from scratch.
- Build an **explanation facility** that answers *how* a conclusion was reached.
- Handle uncertain knowledge using **certainty factors** and **Bayes' theorem**.
- Identify when Forward Chaining is better and when Backward Chaining is better.

# 2. Problem Statement

Imagine a clinic where a junior nurse records a patient's symptoms, and a
computer system suggests what the illness might be.

The system does not learn from data. Instead, a doctor has written down the rules
they use, and the computer applies them.

Our task is:

> **Given a patient's symptoms, work out the most likely diagnosis, and be able
> to explain the reasoning.**

> Important: this is a teaching exercise. A real diagnostic system must be built
> and validated by qualified medical professionals.

# 3. What is an Expert System?

An expert system copies the reasoning of a human expert. It has three parts.

```text
   +-----------------------+
   |    KNOWLEDGE BASE     |   Facts + Rules written by an expert
   +-----------------------+
              |
              v
   +-----------------------+
   |   INFERENCE ENGINE    |   Applies the rules to the facts
   +-----------------------+
              |
              v
   +-----------------------+
   |  EXPLANATION FACILITY |   Says WHY it reached the conclusion
   +-----------------------+
```

The important idea is that the **knowledge is separate from the program**. To
teach the system about a new illness, you add a rule — you do not rewrite the
inference engine.

This is what makes an expert system different from ordinary code.

# 4. Facts

A **fact** is something we currently know to be true.

For our clinic, a fact is a symptom the patient has.

```text
   Patient facts:

   fever          <- the patient has a fever
   cough          <- the patient has a cough
   body_ache      <- the patient has body aches
```

We store the facts in a Python **set**, because a fact is either known or not
known, and order does not matter.

In [ ]:
# The symptoms this patient reported
patient_facts = {"fever", "cough", "body_ache"}

print("Known facts about the patient:")
for fact in sorted(patient_facts):
    print(" -", fact)

print()
print("Number of facts:", len(patient_facts))

# 5. Rules

A **rule** is a piece of expert knowledge written as **IF ... THEN ...**

```text
   IF   fever AND cough AND body_ache
   THEN flu
```

- The **IF** part is called the **conditions** (or the antecedent).
- The **THEN** part is called the **conclusion** (or the consequent).

A rule fires only when **every** condition is a known fact.

Notice that a conclusion can itself become a fact, which may then satisfy another
rule. That is how the system chains short steps into long reasoning.

We store each rule as a Python dictionary.

In [ ]:
rules = [
    {"id": "R1",
     "if": ["fever", "cough", "body_ache"],
     "then": "flu"},

    {"id": "R2",
     "if": ["fever", "rash"],
     "then": "measles"},

    {"id": "R3",
     "if": ["cough", "shortness_of_breath"],
     "then": "asthma"},

    {"id": "R4",
     "if": ["flu", "high_fever"],
     "then": "severe_flu"},

    {"id": "R5",
     "if": ["severe_flu"],
     "then": "admit_to_hospital"},

    {"id": "R6",
     "if": ["flu"],
     "then": "prescribe_rest"},

    {"id": "R7",
     "if": ["sore_throat", "fever"],
     "then": "throat_infection"},
]

print("Knowledge base:\n")
for rule in rules:
    conditions = " AND ".join(rule["if"])
    print(f"{rule['id']}: IF {conditions} THEN {rule['then']}")

# 6. What is Inference?

**Inference** means using rules to turn what we know into something new.

There are two directions we can work in.

```text
   FORWARD CHAINING              BACKWARD CHAINING
   (data driven)                 (goal driven)

   I know: fever, cough          I want to prove: flu
            body_ache
        |                            |
        v                            v
   Which rules can fire?         Which rule concludes flu?
        |                            |
        v                            v
   flu, prescribe_rest           Are its conditions true?
        |                            |
        v                            v
   Keep going until nothing      Prove each condition,
   new appears                   asking sub-questions
```

Forward chaining starts from the **facts** and works towards conclusions.

Backward chaining starts from a **goal** and works back towards the facts.

Both use the same knowledge base. They only differ in where they start.

# 7. Forward Chaining

Forward chaining repeatedly scans every rule. If a rule's conditions are all
known facts, it fires and adds its conclusion to the facts.

It keeps scanning until a full pass adds nothing new. That point is called
**quiescence** — the system has derived everything it possibly can.

```text
   Pass 1: facts = {fever, cough, body_ache}
           R1 fires -> add flu

   Pass 2: facts = {fever, cough, body_ache, flu}
           R6 fires -> add prescribe_rest

   Pass 3: nothing new -> stop
```

## Forward Chaining Algorithm

1. Start with the known facts.
2. Look at every rule in turn.
3. If all of a rule's conditions are known facts, and its conclusion is new, fire it.
4. Add the conclusion to the facts.
5. If a full pass added nothing new, stop.
6. Repeat.

In [ ]:
def forward_chaining(facts, rules):
    # Work on a copy so the original patient facts are not changed
    known = set(facts)

    fired = []          # rules that fired, in order
    trace = []          # how each new fact was derived
    rules_checked = 0

    changed = True
    while changed:
        changed = False

        for rule in rules:
            rules_checked = rules_checked + 1

            # Are all the conditions known?
            all_conditions_met = True
            for condition in rule["if"]:
                if condition not in known:
                    all_conditions_met = False

            # Is the conclusion actually new?
            if all_conditions_met and rule["then"] not in known:
                known.add(rule["then"])
                fired.append(rule["id"])
                trace.append((rule["then"], rule["id"], list(rule["if"])))

                conditions = " AND ".join(rule["if"])
                print(f"Fired {rule['id']}: IF {conditions} THEN {rule['then']}")
                print(f"  New fact added: {rule['then']}")

                changed = True

    return known, fired, trace, rules_checked

# 8. Run Forward Chaining

We will start from the patient's three symptoms and see what the system derives.

In [ ]:
print("===== FORWARD CHAINING EXECUTION =====\n")

print("Starting facts:", sorted(patient_facts))
print()

fc_facts, fc_fired, fc_trace, fc_checked = forward_chaining(patient_facts, rules)

print("\n===== FORWARD CHAINING RESULT =====")
print("All facts now known:", sorted(fc_facts))
print("New facts derived  :", sorted(fc_facts - patient_facts))
print("Rules fired        :", fc_fired)
print("Rule checks made   :", fc_checked)

Notice what happened. R1 fired first and produced `flu`. On the next pass, `flu`
was itself a known fact, which allowed R6 to fire and produce `prescribe_rest`.

That is **chaining**: the conclusion of one rule became the condition of another.

Notice also what did **not** happen. R4 needs `high_fever`, which this patient
does not have, so `severe_flu` was never derived. The system does not guess.

# 9. Backward Chaining

Backward chaining works the other way round. You give it a **goal**, and it asks:

> Can I prove this?

```text
   Goal: severe_flu
     |
     +-- Is it already a known fact? No.
     |
     +-- Which rule concludes severe_flu?  R4
     |
     +-- R4 needs: flu AND high_fever
           |
           +-- Prove flu ......... R1 needs fever, cough, body_ache -> all known, TRUE
           |
           +-- Prove high_fever ... no rule concludes it, not a fact -> FALSE
     |
     +-- One condition failed, so severe_flu is NOT proved
```

This is a **recursive** process: proving a goal means proving its sub-goals, and
each sub-goal is handled exactly the same way.

The big advantage is that backward chaining only touches the rules relevant to
the goal. It never derives facts nobody asked about.

## Backward Chaining Algorithm

1. If the goal is already a known fact, it is proved.
2. Otherwise, find every rule whose conclusion is the goal.
3. For each such rule, try to prove all of its conditions.
4. If all conditions of any rule can be proved, the goal is proved.
5. If no rule works, the goal is not proved.

In [ ]:
def backward_chaining(goal, facts, rules, depth=0, counter=None):
    if counter is None:
        counter = {"checks": 0}

    indent = "  " * depth
    print(f"{indent}Trying to prove: {goal}")

    # Step 1: already known?
    if goal in facts:
        print(f"{indent}  '{goal}' is a known fact -> TRUE")
        return True, counter

    # Step 2: find rules that conclude this goal
    for rule in rules:
        counter["checks"] = counter["checks"] + 1

        if rule["then"] == goal:
            conditions = " AND ".join(rule["if"])
            print(f"{indent}  {rule['id']} could prove it: IF {conditions}")

            # Step 3: try to prove every condition
            all_proved = True
            for condition in rule["if"]:
                proved, counter = backward_chaining(condition, facts, rules,
                                                    depth + 2, counter)
                if not proved:
                    all_proved = False
                    break

            if all_proved:
                print(f"{indent}  All conditions of {rule['id']} hold -> '{goal}' is TRUE")
                return True, counter
            else:
                print(f"{indent}  {rule['id']} failed")

    print(f"{indent}  Nothing proves '{goal}' -> FALSE")
    return False, counter

# 10. Run Backward Chaining

We will ask two questions. First a goal that should succeed, then one that
should fail.

In [ ]:
print("===== BACKWARD CHAINING: does this patient have flu? =====\n")

flu_result, flu_counter = backward_chaining("flu", patient_facts, rules)

print("\nAnswer:", flu_result)
print("Rule checks made:", flu_counter["checks"])

In [ ]:
print("===== BACKWARD CHAINING: does this patient have severe flu? =====\n")

severe_result, severe_counter = backward_chaining("severe_flu", patient_facts, rules)

print("\nAnswer:", severe_result)
print("Rule checks made:", severe_counter["checks"])

Read the second trace carefully. To prove `severe_flu`, the system needed both
`flu` and `high_fever`.

It proved `flu` without difficulty. But nothing in the knowledge base concludes
`high_fever`, and the patient never reported it — so that sub-goal failed, and
the whole goal failed with it.

The system did not guess, and it did not fall back on the diagnosis it already
had. **A missing fact is not the same as a false fact**, and this is one of the
main limitations we return to in section 18.

# 11. The Explanation Facility

The third part of an expert system is the ability to justify itself. A doctor
will not act on a diagnosis the machine cannot defend.

Because forward chaining recorded which rule produced each fact, we can rebuild
the whole chain of reasoning.

```text
   HOW did you conclude prescribe_rest?

   prescribe_rest
      <- R6, because: flu
            flu
               <- R1, because: fever, cough, body_ache
                     fever      (told by the patient)
                     cough      (told by the patient)
                     body_ache  (told by the patient)
```

In [ ]:
def explain(fact, trace, original_facts, depth=0):
    """Print how a fact was derived, following the chain back to the symptoms."""
    indent = "   " * depth

    # Was this something the patient told us?
    if fact in original_facts:
        print(f"{indent}{fact}  (reported by the patient)")
        return

    # Otherwise find the rule that produced it
    for derived_fact, rule_id, conditions in trace:
        if derived_fact == fact:
            print(f"{indent}{fact}")
            print(f"{indent}   <- {rule_id}, because: {', '.join(conditions)}")
            for condition in conditions:
                explain(condition, trace, original_facts, depth + 2)
            return

    print(f"{indent}{fact}  (not derived)")


print("===== EXPLANATION =====\n")

for derived in sorted(fc_facts - patient_facts):
    print("HOW did you conclude:", derived)
    explain(derived, fc_trace, patient_facts)
    print()

This is the feature that separates an expert system from a black box. Every
conclusion can be traced back to the symptoms the patient actually reported and
the rules the doctor actually wrote.

A neural network can give you a diagnosis. It cannot usually give you this.

# 12. Compare Execution Time

For a fair comparison, we will create quieter versions without print statements.

We use `time.perf_counter()` to measure elapsed execution time.

> Because our knowledge base is very small, execution times may be extremely tiny
> and can vary between runs. The comparison demonstrates **how to measure
> performance**, not that one method is always faster.

In [ ]:
import time


def forward_chaining_quiet(facts, rules):
    known = set(facts)
    fired = []
    rules_checked = 0
    changed = True
    while changed:
        changed = False
        for rule in rules:
            rules_checked = rules_checked + 1
            if all(c in known for c in rule["if"]) and rule["then"] not in known:
                known.add(rule["then"])
                fired.append(rule["id"])
                changed = True
    return known, fired, rules_checked


def backward_chaining_quiet(goal, facts, rules, counter=None):
    if counter is None:
        counter = {"checks": 0}
    if goal in facts:
        return True, counter
    for rule in rules:
        counter["checks"] = counter["checks"] + 1
        if rule["then"] == goal:
            if all(backward_chaining_quiet(c, facts, rules, counter)[0]
                   for c in rule["if"]):
                return True, counter
    return False, counter


# Measure forward chaining
start_time = time.perf_counter()
fq_facts, fq_fired, fq_checked = forward_chaining_quiet(patient_facts, rules)
forward_time = time.perf_counter() - start_time

# Measure backward chaining for the same single question
start_time = time.perf_counter()
bq_result, bq_counter = backward_chaining_quiet("flu", patient_facts, rules)
backward_time = time.perf_counter() - start_time

print("Forward Chaining Time :", forward_time, "seconds")
print("Backward Chaining Time:", backward_time, "seconds")

# 13. Compare Work Done

For this practical, we measure work as the **number of rule checks**.

A smaller number means less effort.

> Forward chaining answers every question at once. Backward chaining answers only
> the question you asked. That is why the counts are not directly comparable —
> they are doing different jobs.

In [ ]:
print("FORWARD CHAINING")
print("  Rule checks     :", fq_checked)
print("  Facts derived   :", sorted(fq_facts - patient_facts))
print("  Questions answered: all of them")

print()

print("BACKWARD CHAINING (goal = flu)")
print("  Rule checks     :", bq_counter["checks"])
print("  Facts derived   : none stored, it only answers True/False")
print("  Questions answered: exactly one")

# 14. Final Comparison Table

In [ ]:
results = {
    "Metric": [
        "Starting Point",
        "Direction of Reasoning",
        "Conclusions Produced",
        "Rule Checks",
        "Execution Time (seconds)",
        "Answers One Question or All?",
        "Derives Unrequested Facts?"
    ],
    "Forward Chaining": [
        "Known facts (symptoms)",
        "Facts -> Conclusions",
        ", ".join(sorted(fq_facts - patient_facts)),
        fq_checked,
        forward_time,
        "All",
        "Yes"
    ],
    "Backward Chaining": [
        "A goal (suspected illness)",
        "Goal -> Facts",
        "flu = " + str(bq_result),
        bq_counter["checks"],
        backward_time,
        "One",
        "No"
    ]
}

# Display using pandas if available
try:
    import pandas as pd
    comparison = pd.DataFrame(results)
    display(comparison)
except ImportError:
    for i in range(len(results["Metric"])):
        print(results["Metric"][i])
        print("  Forward :", results["Forward Chaining"][i])
        print("  Backward:", results["Backward Chaining"][i])
        print()

# 15. Forward vs Backward — Conceptual Comparison

| Feature | Forward Chaining | Backward Chaining |
|---|---|---|
| Also Called | Data driven | Goal driven |
| Starts From | Known facts | A goal to prove |
| Works Towards | All possible conclusions | The facts that support the goal |
| Typical Question | "What can I conclude?" | "Can I prove this?" |
| Produces | Every derivable fact | True or False for one goal |
| Wasted Work | May derive facts nobody needs | Almost none |
| Natural Style | Monitoring, alarms | Diagnosis, question answering |
| Implementation | Loop until nothing changes | Recursion |

# 16. When is Forward Chaining Better?

Forward chaining is better when:

1. New facts arrive continuously and we want every consequence immediately.
2. We do not know in advance which question will be asked.
3. The number of possible conclusions is small.

### Example: Patient Monitoring in an ICU

Sensors report a new reading every second. The system should raise **every**
alarm that the new reading justifies, without being asked.

**Example question:**

> Given everything we now know, what is true?

**Recommended method: Forward Chaining**

# 17. When is Backward Chaining Better?

Backward chaining is better when:

1. We have one specific question to answer.
2. The knowledge base is large and most of it is irrelevant to that question.
3. Gathering facts is expensive, so we only want to ask for what we need.

### Example: A Diagnostic Interview

A doctor suspects one illness and wants to confirm it. There is no reason to
derive every other conclusion the rules allow.

Backward chaining also tells you **which fact is missing**, which is exactly the
next question to ask the patient.

**Recommended method: Backward Chaining**

# 18. Reasoning Under Uncertainty

So far every rule has been certain: the conditions hold, so the conclusion is
true. Real medicine is rarely like that.

A cough makes flu **more likely**. It does not make it certain.

## Certainty Factors

The simplest fix is to attach a number from 0 to 1 to each rule, expressing how
strongly the conditions support the conclusion.

## Bayes' Theorem

The proper probabilistic tool is **Bayes' theorem**, which updates a belief when
new evidence arrives.

```text
                  P(evidence | disease) x P(disease)
   P(disease | evidence) = ---------------------------------
                              P(evidence)
```

- **P(disease)** is the **prior** — how common the illness is before we look at
  this patient.
- **P(evidence | disease)** is how likely the symptom is if the patient does have
  the illness.
- **P(disease | evidence)** is the **posterior** — our updated belief.

In [ ]:
def bayes(prior, likelihood, likelihood_if_not):
    """P(disease | evidence) from the prior and the two likelihoods."""
    evidence = likelihood * prior + likelihood_if_not * (1 - prior)
    return (likelihood * prior) / evidence


# A disease that affects 1 person in 100
prior = 0.01

# The test is good: 99% of sick people test positive
likelihood = 0.99

# But 5% of healthy people also test positive
likelihood_if_not = 0.05

posterior = bayes(prior, likelihood, likelihood_if_not)

print("Before the test, chance of having the disease : %.4f" % prior)
print("Test is positive.")
print("After the test, chance of having the disease  : %.4f" % posterior)
print()
print("As a percentage: %.1f%%" % (posterior * 100))

This result surprises most people, and it is the single most useful thing in this
section.

The test is 99% accurate, the patient tested positive, and yet the chance they
actually have the disease is only about 17%.

The reason is the **prior**. The disease is rare, so there are far more healthy
people than sick ones. Five percent of a very large healthy group produces more
false positives than ninety-nine percent of a very small sick group produces true
positives.

> A rule that ignores the prior will confidently give the wrong answer. This is
> why medical expert systems must reason with probabilities, not just IF-THEN
> rules.

# 19. Important Limitation

Rule-based expert systems have real weaknesses that you should be able to name.

- **The knowledge bottleneck.** Every rule must be written by a human expert.
  Building a serious system can take years.
- **No learning.** The system will make the same mistake for ever unless somebody
  edits the rules. It does not improve with use.
- **Brittleness.** Just outside its rules, it fails badly rather than gracefully.
  Our system had no idea what to do with a symptom nobody wrote a rule for.
- **Missing facts are treated as false.** In section 10 the patient was not proved
  to have `severe_flu`. That is not the same as proving they do not have it.
- **Conflicting rules.** If two rules fire and disagree, something must decide
  which wins. That decision is called **conflict resolution**, and our simple
  engine does not do it at all.

Modern systems often combine both approaches: rules for knowledge that is known
and must be auditable, and machine learning for patterns nobody can write down.

# 20. Student Exercise

Try the following:

### Exercise 1
Add `high_fever` to `patient_facts` and run forward chaining again. Which extra
rules fire now?

### Exercise 2
Ask backward chaining to prove `admit_to_hospital` both before and after adding
`high_fever`. Explain the difference in the traces.

### Exercise 3
Add a new rule R8 for a new illness of your choice, then run the system again.
Notice that you did not have to change the inference engine at all.

### Exercise 4
Add a rule whose conclusion contradicts an existing one. Run forward chaining.
What does the system do, and what *should* it do?

### Exercise 5
Compare, for both methods:
- Conclusions produced
- Rule checks
- Execution time

### Exercise 6
Change the prior in the Bayes cell from `0.01` to `0.30` and rerun. Explain why
the same positive test now means something very different.

In [ ]:
# Student Practice Area
# Add a new symptom or a new rule here.

# Example:
# patient_facts.add("high_fever")

# Example:
# rules.append({"id": "R8",
#               "if": ["fever", "joint_pain"],
#               "then": "dengue"})

# Then run:
# facts, fired, trace, checks = forward_chaining(patient_facts, rules)
# print(sorted(facts))

print("Practice area ready!")

# 21. Conclusion

In this practical, we built a rule-based medical expert system and reasoned with
it in two directions.

### Forward Chaining
- Starts from the **known facts**
- Works towards **all** possible conclusions
- Repeats until nothing new appears
- Best when new data arrives and every consequence matters

### Backward Chaining
- Starts from a **goal**
- Works back towards the facts that would support it
- Naturally recursive, and ignores irrelevant rules
- Best when there is one specific question to answer

### The Explanation Facility
- Rebuilds the chain of rules that produced a conclusion
- Turns the system from a black box into something a doctor can check
- Is the main practical advantage of rules over machine learning

### Final Decision

For a diagnostic assistant answering a doctor's specific question,
**Backward Chaining is generally the better choice**, because it asks only for
the facts it actually needs.

For continuous monitoring, where every consequence of new data matters,
**Forward Chaining is the better choice**.

**Key idea:**

> The knowledge lives in the rules, not in the program. Change the rules and the
> system changes its mind — which is exactly what makes an expert system
> maintainable, and exactly what makes it only as good as its rules.